# Notebook 2 — Introduction to Image Classification

**What you will do:**
1. Load the pre-trained YOLOv8 model from the git repo (no internet download).
2. Classify the 3 real parcel images that come with the repo.
3. Read the confidence scores and understand the raw output.
4. See why fine-tuning is needed before Notebook 3.

| Image file | What it shows |
|---|---|
| `cardboard_box.jpg` | Two neat, intact brown parcels tied with twine |
| `damaged_package.jpg` | A crushed FRAGILE-labelled parcel |
| `street_scene.jpg` | A school bus on a road |

> **Note:** The pre-trained model uses ImageNet-1k classes, not parcel classes yet.

In [ ]:
# ── Cell 0: Sync lab materials from GitHub ────────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
from ultralytics import YOLO
from PIL import Image
import pathlib

# ── Load weights from the git repo — no internet download needed ──────────────
WEIGHTS = str(pathlib.Path(LAB) / 'models' / 'yolov8n-cls.pt')
model   = YOLO(WEIGHTS)
print(f'✅ YOLOv8n-cls loaded from {WEIGHTS}')
print(f'   ImageNet-1k classes: {len(model.names)}')

In [ ]:
# ── Verify the 3 sample images are present (from git repo) ───────────────────
IMG_DIR = pathlib.Path(LAB) / 'sample-images'

SAMPLES = ['cardboard_box.jpg', 'damaged_package.jpg', 'street_scene.jpg']
LABELS  = ['Intact parcel',     'Damaged parcel',      'Street scene (bus)']

all_present = True
for fname, label in zip(SAMPLES, LABELS):
    path = IMG_DIR / fname
    if path.exists():
        print(f'  ✅ {fname:<26}  {path.stat().st_size:>8,} bytes  — {label}')
    else:
        print(f'  ❌ {fname:<26}  NOT FOUND at {path}')
        all_present = False

if all_present:
    print('\n✅ All 3 images present — ready to classify')
else:
    raise FileNotFoundError('Sample images missing — re-run the git clone cell above.')

In [ ]:
# ── Run classification on each image and print top-5 scores ───────────────────
for fname, label in zip(SAMPLES, LABELS):
    img_path = str(IMG_DIR / fname)
    results = model(img_path, verbose=False)
    r = results[0]

    top5_idx   = r.probs.top5
    top5_conf  = r.probs.top5conf.tolist()
    top5_names = [r.names[i] for i in top5_idx]

    print(f'\n📦 {label} ({fname})')
    print(f'   Top prediction: {top5_names[0]} ({top5_conf[0]*100:.1f}%)')
    print('   Top-5:')
    for name, conf in zip(top5_names, top5_conf):
        bar = '█' * int(conf * 30)
        print(f'     {name:<30} {bar:<30} {conf*100:5.1f}%')

In [ ]:
# ── Visualise all 3 images side-by-side with their top-1 predictions ──────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, fname, label in zip(axes, SAMPLES, LABELS):
    img_path = str(IMG_DIR / fname)
    results  = model(img_path, verbose=False)
    r        = results[0]
    top1_name = r.names[r.probs.top1]
    top1_conf = r.probs.top1conf.item()

    ax.imshow(mpimg.imread(img_path))
    ax.set_title(
        f'{label}\n→ ImageNet: "{top1_name}" ({top1_conf*100:.1f}%)',
        fontsize=10, pad=8
    )
    ax.axis('off')

plt.suptitle(
    'Pre-trained YOLOv8n-cls — ImageNet predictions (before fine-tuning)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print('\n📌 Observation:')
print('   The model predicts generic ImageNet classes like "carton", "envelope", "school_bus".')
print('   It has no concept of intact / damaged / tampered parcels.')
print('   Notebook 3 will fix this by fine-tuning on your custom parcel dataset.')

In [ ]:
print('\n✅ Notebook 2 complete — proceed to Notebook 3 (fine-tuning)')